# Kalman Filterを用いた線形システムのパラメータ推定

Kalman Filterを用いて、線形システムのパラメータの推定を行う。



## 離散モデル

次の線形モデルの$m[k], D[k]$を測定値の$f[k], a[k], v[k]$から求めることを考える。

$$
f[k] = m[k] \ a[k] + D[k] \  v[k] + w[k]
$$

上式は、慣性 $m[n]$と、粘性 $D[n]$ による運動モデルである。そして、$w[n]$は測定値に含まれるノイズを表した観測雑音であり、平均$0$、分散$\sigma_w^2$の正規分布に従う。

ここでは加速度と速度の測定誤差は無視し、力の観測値に加法的な観測雑音が含まれると仮定する。

この仮定より、$m[n],D[n]$ が変化していない状態で上式の分散を取ると以下のようになる。

$$
V\left[f[k] \right] = V \left[m[k] \ a[k] + D[k] \  v[k] + w[k] \right] = V \left[ w[k] \right] = \sigma_w^2
$$


## 推定モデル

推定すべきパラメータを${\theta}[k]$、 システムの応答値を$z[k]$、システムへの入力を$y[k]$とする。

$$
{\theta}[k] = \begin{bmatrix} m[k] \\ D[k] \end{bmatrix}, \quad z[k] = \begin{bmatrix} a[k] \\ v[k] \end{bmatrix} , \quad y[k] = f[k]
$$

すると、離散モデルは次の形式で記述できる。

$$
y[k] = {\theta}[k]^T z[k] + \eta [k]
$$

${\theta}[k]$は時変であるため、次のランダムウォークするウィナーモデルで表現することができる。<br>
$\eta[k]$は平均$0$、分散$Q$の正規分布に従う。

$$
{\theta}[k+1] = {\theta}[k] + \eta [k]
$$

$Q=\text{diag}(\sigma_m^2, \sigma_D^2)$ であり、$m[k],D[k]$ の変化量を各パラメータの分散で表す。

## Kalman Fitlerによる推定アルゴリズム

### 初期値を次のように設定する

- $\hat{\theta}[0] = \theta_0 \in \mathrm{R}^{2 \times 1}$
    - $\theta_0$は事前にある程度の値が分かればそれを設定する。
    - 事前情報が利用できない場合は $\theta_0 = 0$ とする。
- $P[0] = \text{diag}(\sigma_m^2, \sigma_D^2)$ (2 x 2 の対角行列)
    - 事前情報が利用できない場合、ある程度の大きさに設定する($10^3 \mathrm{I}$など)

### アルゴリズム

各時刻で次の式を上から順番に計算していくことで、推定パラメータ$\hat{\theta}[k]$を求める。

$$
\begin{aligned}
P^{-}[k] &= P[k-1] + Q \\
\epsilon[k] &= y[k] - z[k]^T \hat{\theta}[k-1] \\
g[k] &= \frac{P^{-}[k] \ z[k]}{z[k]^T P^{-}[k] \ z[k] + \sigma_w^2} \\
\hat{\theta}[k] &= \hat{\theta}[k-1] + g[k] \epsilon[k] \\
P[k] &= (\mathrm{I} - g[k] z[k]^T)P^{-}[k]
\end{aligned}
$$

- $P[k]$ : 真値と推定値の誤差の共分散行列
- $P^{-}[k]$ 時刻$k$における誤差共分散行列の推定値 (2 x 2 の行列)
- $\epsilon[k]$ : 観測値と予測観測値の差(イノベーション)
- $g[k]$ : カルマンゲイン ( 2 x 1 の縦ベクトル)